In [0]:
import requests
from pyspark.sql.functions import current_timestamp

In [0]:
dbutils.widgets.text("API_KEY", "")
dbutils.widgets.text("SUBGRAPH_ID", "")

In [0]:
API_KEY = dbutils.widgets.get("API_KEY")
SUBGRAPH_ID = dbutils.widgets.get("SUBGRAPH_ID")

In [0]:
SUBGRAPH_UNIV3_URL = f"https://gateway.thegraph.com/api/{API_KEY}/subgraphs/id/{SUBGRAPH_ID}"
query_variables = {
    "MAINNET_FACTORY_ID": "0x1F98431c8aD98523631AE4a59f267346ea31F984"
}


#factory

In [0]:
factory_query = """
query FactoryQuery($MAINNET_FACTORY_ID: ID!){
    factory(id: $MAINNET_FACTORY_ID ) {
    id
    owner
    poolCount
    txCount
    totalFeesUSD
    totalFeesETH
    totalVolumeUSD
    totalVolumeETH
    totalValueLockedUSD
    totalValueLockedETH
  }
}
"""

In [0]:
response = requests.post(SUBGRAPH_UNIV3_URL, json={"query": factory_query, "variables": query_variables})

In [0]:
df = spark.createDataFrame([response.json()["data"]["factory"]])
df = df.withColumn("_load_ts", current_timestamp())

In [0]:
display(df)

In [0]:
df.write.format("delta").mode("append").saveAsTable("uniswap.bronze.factory")